# 모의고사 08. 매장 매출 예측 (회귀, 결측치·이상치 포함)

> 실제 시험처럼 이론 설명 없이 문제만 순서대로 제시합니다. **90분 타이머를 맞추고** 풀어보세요. 오픈북 허용 문서(numpy/pandas/matplotlib/seaborn/scikit-learn/tensorflow/XGBoost 공식문서)만 참고할 수 있다고 가정합니다.

### 데이터 소개
`data/retail_sales.csv` - 매장 특성(지역, 크기, 광고비, 직원수, 주말여부)으로 매출을 예측합니다. `광고비_만원`에 결측치가, `매출_만원`에 이상치가 의도적으로 섞여 있습니다.

> ⏱️ **[실전 타임어택 가이드]** 데이터 탐색 10분 ➔ 전처리 20분 ➔ 머신러닝 30분 ➔ 딥러닝 30분
> 막히는 부분은 주석으로 남기고 다음 문제로 넘어가는 것이 합격 전략입니다.


## 목차
1교시 데이터 로딩 & EDA · 2교시 데이터 시각화 · 3교시 데이터 전처리 · 4교시 머신러닝 모델링 · 5교시 딥러닝 모델링 · 채점 가이드

In [ ]:
import sys
sys.path.insert(0, '../00_시작하기')
import aice_grader as grader

# 실전처럼 시간 제한(90분)을 두고 풀어보세요. 아래 셀을 실행하면 타이머가 시작됩니다.
timer = grader.ExamTimer(minutes=90)
timer.start()

## 1교시: 데이터 로딩 & EDA

**문제 1.** `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`을 각각 `pd`, `np`, `plt`, `sns`로 불러오고, `../data/retail_sales.csv`을 `sales`로 불러와 shape을 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sales = pd.read_csv('../data/retail_sales.csv')
sales.head()
print(sales.shape)
```

</details>

**문제 2.** `sales`의 결측치 개수를 컬럼별로 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
print(sales.isnull().sum())  # 컬럼별 결측치 개수를 먼저 확인
```

</details>

**문제 3.** `매출_만원`의 boxplot으로 이상치가 있는지 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sns.boxplot(x=sales['매출_만원'])  # 이상치와 사분위수를 확인하기 위해 상자 수염 그림 시각화
plt.show()
```

</details>

## 2교시: 데이터 시각화

**문제 4.** `지역`별 평균 매출을 bar 그래프로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sales.groupby('지역')['매출_만원'].mean().plot(kind='bar')  # groupby+mean으로 만든 Series에 바로 .plot()을 연결해 빠르게 시각화
plt.title('지역별 평균 매출')
plt.show()
```

</details>

**문제 5.** `매장크기_m2`와 `매출_만원`의 관계를 scatterplot으로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sns.scatterplot(data=sales, x='매장크기_m2', y='매출_만원')  # 두 변수 간의 관계를 산점도로 시각화
plt.show()
```

</details>

## 3교시: 데이터 전처리

**문제 6.** `광고비_만원`의 결측치를 중앙값으로 채우세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sales['광고비_만원'] = sales['광고비_만원'].fillna(sales['광고비_만원'].median())  # 결측치를 지정한 값(평균, 중앙값 등)으로 안전하게 대체
```

</details>

**문제 7.** `매출_만원`에서 IQR 기준 이상치를 상한값으로 clip하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
q1, q3 = sales['매출_만원'].quantile([0.25, 0.75])
iqr = q3 - q1
upper = q3 + 1.5 * iqr
sales['매출_만원'] = sales['매출_만원'].clip(upper=upper)  # 상한값/하한값을 벗어나는 이상치를 해당 임계값으로 자름(제한)
sns.boxplot(x=sales['매출_만원'])  # 이상치와 사분위수를 확인하기 위해 상자 수염 그림 시각화
plt.show()
```

</details>

**문제 8.** `지역`, `주말여부`를 `get_dummies(drop_first=True)`로 인코딩하고, `train_test_split(test_size=0.2, random_state=42)`로 분할하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sales_enc = pd.get_dummies(sales, columns=['지역', '주말여부'], drop_first=True)  # 문자로 된 범주형 데이터를 기계가 이해할 수 있는 0과 1 형태(원-핫 인코딩)로 변환
from sklearn.model_selection import train_test_split
X = sales_enc.drop(columns=['매출_만원'])
y = sales_enc['매출_만원']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # 데이터를 모델 학습용(train)과 검증용(test)으로 분리
print(X_train.shape)
```

</details>

## 4교시: 머신러닝 모델링

**문제 9.** `LinearRegression`과 `RandomForestRegressor(random_state=42)`를 각각 학습시켜 R2를 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
lr = LinearRegression()  # 선형 회귀 모델 생성
lr.fit(X_train, y_train)
print('Linear R2:', r2_score(y_test, lr.predict(X_test)))  # 회귀 모델의 설명력(R-squared) 계산 (1에 가까울수록 우수)
rf = RandomForestRegressor(random_state=42)  # 랜덤 포레스트 회귀 모델 생성
rf.fit(X_train, y_train)
print('RF R2:', r2_score(y_test, rf.predict(X_test)))  # 회귀 모델의 설명력(R-squared) 계산 (1에 가까울수록 우수)
```

</details>

## 5교시: 딥러닝 모델링

**문제 10.** `StandardScaler`로 스케일링한 뒤 딥러닝 회귀 모델을 만들어 학습시키고 R2를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
scaler = StandardScaler()  # 데이터를 평균 0, 표준편차 1이 되도록 스케일링 (거리 기반, 딥러닝 알고리즘에 필수)
X_train_s = scaler.fit_transform(X_train)  # 훈련 데이터로 스케일링 기준을 학습(fit)하고 변환(transform)까지 수행
X_test_s = scaler.transform(X_test)  # 훈련 데이터에서 학습된 기준을 그대로 적용하여 평가 데이터를 변환 (데이터 누수 방지)
model = Sequential()  # 딥러닝 층(Layer)을 순서대로 쌓기 위한 기본 틀(모델) 생성
model.add(Dense(16, activation='relu', input_shape=(X_train_s.shape[1],)))  # 이전 층의 모든 노드와 연결되는 완전 연결층(Dense Layer) 추가
model.add(Dense(8, activation='relu'))  # 이전 층의 모든 노드와 연결되는 완전 연결층(Dense Layer) 추가
model.add(Dense(1))  # 이전 층의 모든 노드와 연결되는 완전 연결층(Dense Layer) 추가
model.compile(optimizer='adam', loss='mse', metrics=['mae'])  # 모델 학습 과정(최적화 기법, 손실 함수, 평가지표) 설정
es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)  # 성능이 개선되지 않으면 무의미한 학습을 일찍 멈춤
model.fit(X_train_s, y_train, epochs=50, batch_size=16, validation_data=(X_test_s, y_test), callbacks=[es], verbose=0)
print(r2_score(y_test, model.predict(X_test_s).flatten()))  # 회귀 모델의 설명력(R-squared) 계산 (1에 가까울수록 우수)
```

</details>

In [ ]:
# 문제를 다 풀었다면 아래 셀을 실행해 소요 시간을 확인하세요.
timer.finish()

## 채점 가이드 (100점 만점 배점표)

이 모의고사는 총 10문제, 100점 만점입니다. 각 문제를 정답과 비교해 맞았으면 배점만큼, 틀렸으면 0점으로 스스로 채점해보세요.

| 문항 | 배점 |
|---|---|
| 문제 1 | 10점 |
| 문제 2 | 10점 |
| 문제 3 | 10점 |
| 문제 4 | 10점 |
| 문제 5 | 10점 |
| 문제 6 | 10점 |
| 문제 7 | 10점 |
| 문제 8 | 10점 |
| 문제 9 | 10점 |
| 문제 10 | 10점 |
| **합계** | **100점** |

> 💡 **합격 기준: 80점 이상** (실제 AICE Associate와 동일한 기준입니다)

### 자동으로 기록하며 채점하고 싶다면

`00_시작하기/aice_grader.py`의 `MockExamGrader`를 사용하면 점수를 자동으로 합산해줍니다.

```python
import aice_grader as grader

g = grader.MockExamGrader("모의고사08_매장매출예측")
g.check(1, points=10, is_correct=True)   # 문제 1을 맞았으면 True, 틀렸으면 False
g.check(2, points=10, is_correct=True)
# ... 문제 수만큼 반복 ...
g.report()   # 최종 점수와 합격 여부를 출력
```

### 개념 체크리스트

다 풀었다면 아래 체크리스트로 한 번 더 점검해보세요.

- [ ] 라이브러리를 정확한 alias(pd, np, plt, sns 등)로 불러왔는가
- [ ] 결측치와 이상치를 모두 처리했는가
- [ ] clip으로 이상치를 처리한 이유(제거보다 정보손실이 적음)를 이해했는가
- [ ] 범주형 컬럼을 인코딩했는가
